# Time-Series Preprocessing

End-to-end cleaning of the CMI accelerometer time-series dataset. 

 Pipeline:

1. **Load & inspect** the raw series.
2. **Missing values** — sanity check on the signal we clean.
3. **Column selection** — keep the signal channels + id + target, drop constant metadata.
4. **Non-wear ("wristband off") filtering** — drop series that are not fully worn.
5. **Physically-impossible spike removal** — drop series whose `enmo` contains
   artifact spikes.
6. **Export** the cleaned series list to `TS/processed_data/`.

In [12]:
import gzip, pickle, zipfile, os
import numpy as np
import pandas as pd

# The raw dataset ships as a gzipped pickle inside a zip in base_data/.
# Load straight from the zip so there is no manual unzip step to keep in sync.
ZIP_PATH = "../../base_data/cmi_timeseries_dataset_dm2_25_26.zip"
MEMBER   = "CMI_timeseries_dataset.pkl.gz"

with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open(MEMBER) as f:
        with gzip.open(f, "rb") as g:
            data = pickle.load(g)

print(f"Series loaded : {len(data)}")
print(f"Columns       : {list(data[0].columns)}")
lengths = np.array([len(df) for df in data])
print(f"Series length : min {lengths.min()}, max {lengths.max()}, "
      f"all equal = {len(set(lengths)) == 1}")

Series loaded : 4437
Columns       : ['X', 'Y', 'Z', 'enmo', 'anglez', 'non-wear_flag', 'light', 'battery_voltage', 'weekday', 'quarter', 'relative_date_PCIAT', 'id', 'sii_binary']
Series length : min 200, max 200, all equal = True


## 1. Missing values

In [13]:
# enmo is the signal we clean downstream; confirm there are no NaNs to handle first.
E_all = np.array([df['enmo'].values for df in data])
print(f"Total NaN in enmo : {np.isnan(E_all).sum()}")
print(f"Series with NaN   : {(np.isnan(E_all).sum(axis=1) > 0).sum()} / {len(data)}")

Total NaN in enmo : 0
Series with NaN   : 0 / 4437


## 2. Column selection

We keep the **signal channels** plus the subject **id** and the **target**, and drop
metadata that is *constant within a series*. Because each series is a single
subject-day, `weekday`, `quarter` and `relative_date_PCIAT` never vary across the 200
epochs, so they carry no within-series information.

`non-wear_flag` is kept only as a **working column** for the next step; it is dropped
at export because it is identically `0` once we keep only fully-worn series.

In [14]:
SIGNAL_COLS = ['X', 'Y', 'Z', 'enmo', 'anglez']
ID_COLS     = ['id', 'sii_binary']            # subject id + classification target
OUTPUT_COLS = SIGNAL_COLS + ID_COLS           # what the exported series will contain

# Retained transiently for the non-wear filter, then dropped at export:
WORK_COLS = OUTPUT_COLS + ['non-wear_flag']

dropped = [c for c in data[0].columns if c not in WORK_COLS]
print("Keep (exported)      :", OUTPUT_COLS)
print("Keep (working only)  : ['non-wear_flag']")
print("Drop (constant meta) :", dropped)

Keep (exported)      : ['X', 'Y', 'Z', 'enmo', 'anglez', 'id', 'sii_binary']
Keep (working only)  : ['non-wear_flag']
Drop (constant meta) : ['light', 'battery_voltage', 'weekday', 'quarter', 'relative_date_PCIAT']


## 3. Non-wear ("wristband off") filtering

`non-wear_flag` is a value in `[0, 1]` giving the **fraction of each ~7-minute epoch
the device was off the wrist**. Off-body epochs are not real physiological
measurements — they are flat/garbage readings that would contaminate any downstream
model.

Rather than try to patch individual off-body epochs, we keep only series that were
**fully worn all day** (`non-wear_flag == 0` for every epoch). This is the principled
"error" handling for this dataset: the non-wear segments are *labelled*, so we drop
the affected series outright instead of mixing real signal with sensor-off artifacts.
It costs only ~4% of the data.

In [15]:
nw = np.array([df['non-wear_flag'].values for df in data])   # (n_series, 200)
frac_off   = (nw > 0).mean(axis=1)                           # fraction of epochs w/ any non-wear
fully_worn = frac_off == 0

print(f"Fully-worn series      : {fully_worn.sum()} / {len(data)} ({fully_worn.mean():.1%})")
print(f"Dropped (any non-wear) : {(~fully_worn).sum()}")

worn_idx  = np.where(fully_worn)[0]      # indices into the ORIGINAL `data`
data_worn = [data[i] for i in worn_idx]

Fully-worn series      : 4258 / 4437 (96.0%)
Dropped (any non-wear) : 179


## 4. Physically-impossible spike removal  *(replaces the Hampel filter)*

### Why not a Hampel filter?

A Hampel filter flags a point when it deviates from its local rolling median by more
than `n_sigma · 1.4826 · MAD`. That assumes the signal is **locally smooth** with
**Gaussian-ish noise**, and treats every large deviation as corruption.

`enmo` violates both assumptions — it is a **sparse, heavy-tailed activity signal**:
mostly near-zero (rest/sleep), punctuated by genuine movement bursts. Two failure
modes follow:

- **MAD collapses in rest windows** → the threshold shrinks toward `0`, so trivial
  fluctuations get flagged as outliers.
- **Real activity looks anomalous** → every movement burst exceeds the local median
  by far more than `n_sigma · MAD`.

Empirically, a Hampel filter (window 7, `n_sigma` 3) flags **~7–12% of all points**;
and using "any flagged point" to drop whole *series* removes **74–96% of the dataset**,
because essentially every series contains a movement burst. The flag simply cannot
separate a **physically impossible artifact** from **normal vigorous activity** — it
discards the one thing that distinguishes them: **absolute magnitude**. → *Rejected.*

### What we do instead

Each epoch is a **~7-minute average** of the raw accelerometer, so `enmo` has a hard
physiological ceiling: even sustained vigorous exercise rarely averages above ~0.5 g,
and a 7-minute average **cannot** reach ~1.5 g from human wrist motion — such values are
sensor **saturation / clipping** artifacts. (Here the two largest values are an
*identical* 7.03 g in two unrelated subjects — a textbook clipping ceiling.) One such
spike, fed into normalization or any distance/gradient computation, silently corrupts
everything downstream.

So we score each **series** by the severity of its worst spike using two signals:

- **`peak`** — the maximum `enmo` in the series (absolute magnitude).
- **`isolation`** — `peak / max(immediate neighbours)`: how far the point towers over
  its surroundings. A glitch spikes *alone* (ratio ≫ 1); genuine activity is *sustained*
  across several epochs (ratio ≈ 1). The isolation test is what protects real bursts.

A series is removed if **either** condition holds:

1. **`peak > HARD_CEILING`** — unconditional. A value this large cannot be a 7-minute
   average under *any* movement pattern, so its shape is irrelevant: it is saturation /
   clipping. This clause is essential — one of the two 7.03 g clips here is *preceded*
   by an elevated epoch (1.89 g), giving it an isolation ratio of only ~3.7; the
   isolation test alone would let it survive, so the hard ceiling guarantees both clips
   are removed.
2. **`peak > PHYS_CEILING` and `isolation > ISO_MIN`** — an isolated, physically-
   implausible spike (a glitch), while sparing *sustained* high-activity series whose
   peak is elevated but whose neighbours are too (ratio ≈ 1).

In [16]:
HARD_CEILING = 3.0    # g  — impossible as a 7-min average under ANY motion -> always remove
PHYS_CEILING = 1.5    # g  — non-physiological as averaged enmo; remove if also an isolated spike
ISO_MIN      = 5.0    # peak must tower >5x over its neighbours to count as an isolated spike

E = np.array([df['enmo'].values for df in data_worn])     # (n_worn, 200)

# isolation: how much each point exceeds the LARGER of its two neighbours
pad   = np.pad(E, ((0, 0), (1, 1)), mode='edge')          # edge-replicate so ends have neighbours
neigh = np.maximum(pad[:, :-2], pad[:, 2:])               # max(left, right) per point
ratio = E / np.maximum(neigh, 1e-3)                       # 1e-3 guards divide-by-zero

peak_val   = E.max(axis=1)                                # worst enmo per series
peak_t     = E.argmax(axis=1)                             # epoch index of that peak
peak_ratio = ratio[np.arange(len(E)), peak_t]             # isolation of the peak

# Two-tier rule: (1) unconditional hard ceiling for clear saturation/clipping, OR
#                (2) physically-implausible AND isolated spike (a glitch).
artifact = (peak_val > HARD_CEILING) | \
           ((peak_val > PHYS_CEILING) & (peak_ratio > ISO_MIN))
print(f"Series flagged as artifact-contaminated: {artifact.sum()} / {len(data_worn)}")

Series flagged as artifact-contaminated: 6 / 4258


**Inspection shortlist.** Every series whose peak reaches the physical ceiling, so the
removal decision is auditable. `removed = True` means the peak is *also* an isolated
spike; `removed = False` rows are sustained high-activity (kept).

In [17]:
ids = np.array([df['id'].iloc[0]         for df in data_worn])
sii = np.array([df['sii_binary'].iloc[0] for df in data_worn])

shortlist = pd.DataFrame({
    'series_idx': worn_idx,               # index in the ORIGINAL `data`
    'id':         ids,
    'sii':        sii,
    'peak_enmo':  peak_val.round(3),
    'peak_epoch': peak_t,
    'isolation':  peak_ratio.round(1),
    'removed':    artifact,
}).query('peak_enmo > @PHYS_CEILING').sort_values('peak_enmo', ascending=False)

print(shortlist.to_string(index=False))

 series_idx   id  sii  peak_enmo  peak_epoch  isolation  removed
       3012 1044    0      7.034         150  52.900002     True
        561 2064    1      7.028          60   3.700000     True
       1862  858    0      2.520         155 195.199997     True
        119 3556    1      2.366          91   2.000000    False
       4099 2657    0      2.181          25   1.200000    False
       3244 2657    0      1.878          84   1.300000    False
       4148 3925    0      1.811         154   1.400000    False
       3306 3925    0      1.676         136   1.100000    False
       1065 2657    0      1.664          98   2.100000    False
        288  978    0      1.641          48   1.800000    False
       1232 2657    0      1.636          88   1.700000    False
       2037 3556    1      1.630         128   1.400000    False
       1611 2657    0      1.579          88   2.200000    False
       3341  382    0      1.559         112   9.800000     True
        829 2657    0    

In [18]:
data_clean = [df for df, bad in zip(data_worn, artifact) if not bad]
print(f"After spike removal: {len(data_clean)} series (removed {int(artifact.sum())})")

After spike removal: 4252 series (removed 6)


## 5. Export cleaned dataset

Apply the final column selection (drop `non-wear_flag` — identically 0 after the wear
filter — and the constant metadata) and write the cleaned list of per-series DataFrames
as a gzipped pickle to `TS/processed_data/`, mirroring CMI's `processed_data/`.

In [19]:
OUT_DIR  = "../processed_data"
OUT_PATH = os.path.join(OUT_DIR, "ts_preprocessed.pkl.gz")
os.makedirs(OUT_DIR, exist_ok=True)

# Final column selection: signal channels + id + target.
data_out = [df[OUTPUT_COLS].reset_index(drop=True) for df in data_clean]

with gzip.open(OUT_PATH, "wb") as f:
    pickle.dump(data_out, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Wrote {len(data_out)} series -> {OUT_PATH}")
print(f"Columns per series : {list(data_out[0].columns)}")
print(f"Pipeline funnel    : {len(data)} raw "
      f"-> {len(data_worn)} worn -> {len(data_clean)} clean")
print(f"Total series dropped: {round(100*(1-len(data_clean)/len(data)),2)}%")

Wrote 4252 series -> ../processed_data/ts_preprocessed.pkl.gz
Columns per series : ['X', 'Y', 'Z', 'enmo', 'anglez', 'id', 'sii_binary']
Pipeline funnel    : 4437 raw -> 4258 worn -> 4252 clean
Total series dropped: 4.17%
